# Tuning de Hiperparâmetros — XGBoost

Notebook para explorar combinações de hiperparâmetros do XGBClassifier.

**Fluxo:**
1. Carrega dados processados
2. Divide em treino / validação / teste (estratificado)
3. Testa combinações via `ParameterGrid` na validação (F1 Score)
4. Retreina com melhores params no treino completo
5. Avalia no teste holdout

Ajuste o `PARAM_GRID` e reexecute as cells a partir da cell 4 a cada iteração.

In [1]:
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    f1_score,
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    matthews_corrcoef,
)
from sklearn.model_selection import train_test_split, ParameterGrid
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder
from xgboost import XGBClassifier

from config import (
    PROCESSED_TRAIN_FILE,
    TARGET_COL,
    NUMERIC_FEATURES,
    CATEGORICAL_FEATURES,
)

print("Imports OK")

Imports OK


## 1. Carregar dados e dividir em treino / validação / teste

In [2]:
# =============================================
# HIPERPARÂMETROS DE SPLIT — ajuste se quiser
# =============================================
RANDOM_STATE = 42
TEST_SIZE = 0.20
VALIDATION_SIZE = 0.10

# Carregar dados processados
df = pd.read_csv(PROCESSED_TRAIN_FILE)
print(f"Dataset: {df.shape[0]:,} linhas × {df.shape[1]} colunas")
print(f"Distribuição do target:")
print(df[TARGET_COL].value_counts().rename({1: "Admissão (1)", -1: "Desligamento (-1)"}))

# Separar features e target
feature_cols = list(NUMERIC_FEATURES) + list(CATEGORICAL_FEATURES)
X = df[feature_cols].copy()
y = df[TARGET_COL].map({1: 1, -1: 0})  # XGBoost espera labels >= 0

# Split 1: treino_completo (80%) + teste (20%)
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)

# Split 2: treino (90% do treino_completo) + validação (10%)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full,
    test_size=VALIDATION_SIZE, random_state=RANDOM_STATE, stratify=y_train_full
)

print(f"\nTamanhos:")
print(f"  Treino       : {X_train.shape[0]:>10,}")
print(f"  Validação    : {X_val.shape[0]:>10,}")
print(f"  Treino compl.: {X_train_full.shape[0]:>10,}")
print(f"  Teste        : {X_test.shape[0]:>10,}")

Dataset: 738,837 linhas × 19 colunas
Distribuição do target:
saldomovimentacao
Desligamento (-1)    377254
Admissão (1)         361583
Name: count, dtype: int64

Tamanhos:
  Treino       :    531,962
  Validação    :     59,107
  Treino compl.:    591,069
  Teste        :    147,768


## 2. Preprocessador

In [3]:
numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="desconhecido")),
    ("encoder", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)),
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, list(NUMERIC_FEATURES)),
        ("cat", categorical_pipeline, list(CATEGORICAL_FEATURES)),
    ],
    remainder="drop",
)

print("Preprocessador montado.")

Preprocessador montado.


## 3. Função de Grid Search

Função reutilizável que recebe um `param_grid`, executa o grid search na validação e exibe o ranking.
Chame quantas vezes quiser com parâmetros diferentes — cada execução fica registrada como justificativa.

In [4]:
def run_grid_search(param_grid: dict) -> tuple[dict, float, pd.DataFrame]:
    """
    Executa grid search manual na validação e exibe o ranking dos resultados.

    Parameters:
        param_grid: Dicionário com hiperparâmetros a testar (formato ParameterGrid).

    Returns:
        Tupla (melhores_params, melhor_f1, dataframe_ranking).
    """
    grid = list(ParameterGrid(param_grid))
    total = len(grid)
    print(f"Total de combinações a testar: {total}")
    print("=" * 80)

    results = []
    best_score = -np.inf
    best_params = None

    for i, params in enumerate(grid, 1):
        clf = XGBClassifier(
            **params,
            objective="binary:logistic",
            eval_metric="logloss",
            n_jobs=-1,
            random_state=RANDOM_STATE,
        )

        pipe = Pipeline(steps=[
            ("preprocessor", preprocessor),
            ("classifier", clf),
        ])

        pipe.fit(X_train, y_train)
        y_val_pred = pipe.predict(X_val)
        score = f1_score(y_val, y_val_pred)

        marker = " *** NOVO MELHOR" if score > best_score else ""
        print(f"[{i:>{len(str(total))}}/{total}] F1: {score:.4f}{marker} | {params}")

        results.append({"f1_val": score, **params})

        if score > best_score:
            best_score = score
            best_params = params

    print("=" * 80)
    print(f"\nMelhores hiperparâmetros: {best_params}")
    print(f"Melhor F1 (validação): {best_score:.4f}")

    # Ranking
    df_results = pd.DataFrame(results).sort_values("f1_val", ascending=False).reset_index(drop=True)
    df_results.index += 1
    df_results.index.name = "rank"
    display(df_results)

    return best_params, best_score, df_results

print("Função run_grid_search() definida.")

Função run_grid_search() definida.


## 4. Execuções do Grid Search

Cada cell abaixo é uma execução com parâmetros diferentes.
Duplique a cell, ajuste o grid e execute novamente — o histórico fica registrado.

In [5]:
# =============================================
# EXECUÇÃO 1 — Exploração inicial ampla
# =============================================
best_params, best_score, df_results = run_grid_search({
    "n_estimators": [300, 500],
    "max_depth": [4, 6, 8],
    "learning_rate": [0.05, 0.1],
    "min_child_weight": [1, 5],
    "subsample": [0.8],
    "colsample_bytree": [0.8],
})

Total de combinações a testar: 24
[ 1/24] F1: 0.5810 *** NOVO MELHOR | {'colsample_bytree': 0.8, 'learning_rate': 0.05, 'max_depth': 4, 'min_child_weight': 1, 'n_estimators': 300, 'subsample': 0.8}
[ 2/24] F1: 0.5814 *** NOVO MELHOR | {'colsample_bytree': 0.8, 'learning_rate': 0.05, 'max_depth': 4, 'min_child_weight': 1, 'n_estimators': 500, 'subsample': 0.8}
[ 3/24] F1: 0.5820 *** NOVO MELHOR | {'colsample_bytree': 0.8, 'learning_rate': 0.05, 'max_depth': 4, 'min_child_weight': 5, 'n_estimators': 300, 'subsample': 0.8}
[ 4/24] F1: 0.5807 | {'colsample_bytree': 0.8, 'learning_rate': 0.05, 'max_depth': 4, 'min_child_weight': 5, 'n_estimators': 500, 'subsample': 0.8}
[ 5/24] F1: 0.5899 *** NOVO MELHOR | {'colsample_bytree': 0.8, 'learning_rate': 0.05, 'max_depth': 6, 'min_child_weight': 1, 'n_estimators': 300, 'subsample': 0.8}
[ 6/24] F1: 0.5921 *** NOVO MELHOR | {'colsample_bytree': 0.8, 'learning_rate': 0.05, 'max_depth': 6, 'min_child_weight': 1, 'n_estimators': 500, 'subsample': 0.8

,f1_val,colsample_bytree,learning_rate,max_depth,min_child_weight,n_estimators,subsample
rank,,,,,,,
1,0.609736,0.8,0.10,8,5,500,0.8
2,0.608315,0.8,0.10,8,1,500,0.8
3,0.604530,0.8,0.10,8,1,300,0.8
4,0.604366,0.8,0.10,8,5,300,0.8
5,0.604340,0.8,0.05,8,5,500,0.8
6,0.604242,0.8,0.05,8,1,500,0.8
7,0.600530,0.8,0.05,8,1,300,0.8
8,0.599535,0.8,0.10,6,1,500,0.8
9,0.599491,0.8,0.10,6,5,500,0.8


## 5. Treinar modelo final no treino completo e avaliar no teste holdout

Quando encontrar os melhores parâmetros, execute esta cell para treinar no treino completo e avaliar no teste.

In [ ]:
# Treina com os melhores hiperparâmetros no treino COMPLETO
print(f"Treinando modelo final com: {best_params}")
print(f"Treino completo: {X_train_full.shape[0]:,} registros")

final_clf = XGBClassifier(
    **best_params,
    objective="binary:logistic",
    eval_metric="logloss",
    n_jobs=-1,
    random_state=RANDOM_STATE,
)

final_pipe = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", final_clf),
])

final_pipe.fit(X_train_full, y_train_full)
y_test_pred = final_pipe.predict(X_test)

target_names = ["Desligamento (-1)", "Admissão (1)"]

print("\n" + "=" * 80)
print("AVALIAÇÃO NO TESTE HOLDOUT")
print("=" * 80)
print(f"  F1 Score (binary)    : {f1_score(y_test, y_test_pred):.4f}  ← métrica principal")
print(f"  F1 Score (macro)     : {f1_score(y_test, y_test_pred, average='macro'):.4f}")
print(f"  F1 Score (weighted)  : {f1_score(y_test, y_test_pred, average='weighted'):.4f}")
print(f"  Accuracy             : {accuracy_score(y_test, y_test_pred):.4f}")
print(f"  Balanced Accuracy    : {balanced_accuracy_score(y_test, y_test_pred):.4f}")
print(f"  MCC                  : {matthews_corrcoef(y_test, y_test_pred):.4f}")
print("-" * 80)
print("Classification Report:")
print(classification_report(y_test, y_test_pred, target_names=target_names, digits=4))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_test_pred))

## 6. Copie os melhores parâmetros para o `config.py`

Quando estiver satisfeito com os resultados, copie o `best_params` para o `XGBOOST_PARAM_GRID` do `config.py` (com listas de um só elemento) e rode `python main.py`.

In [ ]:
print("Cole no config.py XGBOOST_PARAM_GRID:")
print()
print("XGBOOST_PARAM_GRID = {")
for k, v in best_params.items():
    print(f'    "{k}": [{repr(v)}],')
print("}")